# Bias in Machine Learning: Hands-on Implementation

## What This Notebook Covers
This notebook is the practical, code-driven counterpart to the **Bias in Machine Learning README**. Instead of just studying bias theoretically, we will build a deliberately oversimplified linear model from scratch, evaluate its Mean Squared Error (MSE), and compare it against a trained Scikit-learn model to analyze the difference in prediction error.

## What You Will Accomplish
- Describe what bias means for a machine learning model and trace its origins.
- Identify the difference between a high-bias (underfit) model and a low-bias model within the Bias-Variance Tradeoff.
- Build a NumPy-based linear prediction model with frozen, hand-picked parameters and compute its MSE manually.
- Train, evaluate, and interpret a Scikit-learn `LogisticRegression` classifier on multi-dimensional datasets.
- Audit target class balance and coordinate plots to see how feature count updates bias.
- Analyze how experimental parameters like `test_size` affect performance metrics.

## Before You Start (Prerequisites)
- Comfort writing basic Python (variables, loops, functions).
- Familiarity with NumPy matrices and Pandas DataFrames.
- Understanding of standard train-test validation splits.

## About the Dataset
This notebook uses the benchmark **Iris flower dataset**, containing 150 instances of flowers split evenly across three species: *setosa*, *versicolor*, and *virginica*. For every flower, we have four numeric measurements:
- Sepal length (cm)
- Sepal width (cm)
- Petal length (cm)
- Petal width (cm)

We load it directly using `sklearn.datasets.load_iris`. If you want to explore the dataset outside this notebook, it is also archived on Kaggle:
**https://www.kaggle.com/datasets/uciml/iris**

---

## 1. Setup & Workspace Preparation

### WHY?
We must import core libraries for matrix mathematics, tabular calculations, visualization, and model fitting. Loading these upfront guarantees a clean working directory.

### HOW?
We import NumPy, Pandas, Matplotlib, Seaborn, and the necessary Scikit-learn datasets, linear estimators, and metric tables.

In [ ]:
# Import NumPy for fast matrix arithmetic and manual MSE loss functions
import numpy as np

# Import Pandas to display data statistics within dataframes
import pandas as pd

# Import Matplotlib and Seaborn for plotting target class boundaries
import matplotlib.pyplot as plt
import seaborn as sns

# Load standard iris dataset classifier tools
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("All libraries imported successfully.")

## 2. Dataset Loading & Exploration

### WHY?
To analyze bias, we need to inspect the dimensions and statistics of the data. Loading the features and mapping target codes to human-readable labels makes inspection straightforward.

### HOW?
We fetch the dataset object, structure features inside a Pandas DataFrame, map targets to species names, and print statistical summaries.

In [ ]:
# Load iris dataset dictionary from sklearn
iris = load_iris()

# Wrap measurements inside a dataframe, applying original column names
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# Append target codes (0, 1, 2) to the dataframe
df['species'] = iris.target

# Map species target integers to string labels to improve readability
df['species_name'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Peek at the first 5 records
print("First 5 rows of the dataset:")
print(df.head())

# Print dataframe properties to check coordinates and data types
print("\nDataset info:")
print(df.info())

# Print statistical characteristics (mean, std, min, max)
print("\nSummary statistics:")
print(df.describe())

# Summarize data constraints
print("\nNumber of samples:", df.shape[0])
print("Number of features:", len(iris.feature_names))
print("Feature names:", list(iris.feature_names))
print("Target variable: 'species' (0 = setosa, 1 = versicolor, 2 = virginica)")
print("Target classes:", list(iris.target_names))

## 3. Data Auditing & Feature Isolation

### WHY?
We must audit the dataset for missing data or duplicates to ensure input quality. Separating the feature matrix ($X$) from the target vector ($y$) is necessary before fitting models.

### HOW?
We call `.isnull().sum()` and `.duplicated().sum()` to inspect the DataFrame, and split features from the target species label.

In [ ]:
# Verify missing value counts per column
print("Missing values per column:")
print(df.isnull().sum())

# Count duplicate rows in the dataset
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Isolate features (capital X matrix) from output targets (lowercase y vector)
X = df[iris.feature_names]
y = df['species']

print("\nShape of X (features):", X.shape)
print("Shape of y (target):", y.shape)

## 4. Part 2: Building a Toy Model From Scratch

### WHY?
Writing a prediction model manually in NumPy—using frozen parameters that we pick by hand rather than letting an algorithm learn them—provides a clear demonstration of a high-bias (underfit) model.

### HOW?
We define a linear function `predict(x, weight, bias)` and an MSE error function `mean_squared_error_scratch(y_true, y_pred)`. We pick `sepal length` as the single feature input, assign manual parameters (`weight = 0.5`, `bias = -1.5`), and evaluate predictions.

In [ ]:
# Set random seed to make calculations deterministic
np.random.seed(42)

# Linear prediction function representing our manual linear model
def predict(x, weight, bias):
    return weight * x + bias

# Compute Mean Squared Error (MSE)
def mean_squared_error_scratch(y_true, y_pred):
    errors = y_true - y_pred
    squared_errors = errors ** 2
    return np.mean(squared_errors)

# Select a single input feature column
feature_values = X['sepal length (cm)'].values

# Isolate target labels as floats
actual_values = y.values.astype(float)

# Assign manual parameters by hand; these are NOT fit to the dataset
manual_weight = 0.5
manual_bias = -1.5

# Generate predictions using our manual model
predictions_scratch = predict(feature_values, manual_weight, manual_bias)

# Calculate errors
errors_scratch = actual_values - predictions_scratch

# Calculate Mean Squared Error
mse_scratch = mean_squared_error_scratch(actual_values, predictions_scratch)

## 5. Analyzing Toy Model Performance

### WHY?
We print sample predictions and errors alongside the overall MSE score to analyze the directional errors and systematic offsets characteristic of high bias.

### HOW?
We print the first 10 predictions, actual targets, calculated errors, and print the overall dataset MSE.

In [ ]:
# Print the first 10 prediction values
print("First 10 predictions:", np.round(predictions_scratch[:10], 2))

# Print true targets
print("First 10 actual values:", actual_values[:10])

# Print errors
print("First 10 errors:", np.round(errors_scratch[:10], 2))

# Print calculated MSE score
print("\nMean Squared Error (hand-tuned model):", round(mse_scratch, 4))

# Explain why the calculated error represents high bias
print("\nWhat this tells us:")
print("Every prediction above comes from the SAME fixed formula, regardless of which")
print("flower we look at - the model never adjusted itself to the data. Because the")
print("true labels only take the values 0, 1, or 2, an MSE of roughly", round(mse_scratch, 2),
      "is large in relative terms.")
print("This gap between predictions and reality won't shrink by adding more flowers,")
print("because the formula's weight and bias are frozen. That permanent, non-shrinking")
print("gap is exactly what we mean by HIGH BIAS, and the resulting poor fit is UNDERFITTING.")

## 6. Part 3: Training an Adaptable Classifier

### WHY?
To show how learning resolves bias, we train a Scikit-learn `LogisticRegression` classifier on all four feature columns. Letting the algorithm optimize parameters dynamically allows it to fit the patterns in the data, reducing systematic error.

### HOW?
We split features and targets ($80\%$ train, $20\%$ test) with a fixed seed. We train `LogisticRegression(max_iter=200)` on the training set, generate predictions on the test set, and evaluate the model using accuracy, a classification report, and a confusion matrix.

In [ ]:
# Split features/targets into train (80%) and test (20%) datasets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize classifier, setting max_iter to ensure convergence
log_reg = LogisticRegression(max_iter=200)

# Train model on features to optimize parameters
log_reg.fit(X_train, y_train)

# Generate predictions on test data
y_pred = log_reg.predict(X_test)

# Evaluate accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

# Print precision, recall, and F1 metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Print confusion matrix mapping actual vs. predicted labels
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Analyze why the trained model performs better
print("\nHow this compares to our hand-tuned model:")
print("- Our NumPy model used ONE feature with frozen, hand-picked numbers and produced")
print("  a large, unchanging Mean Squared Error - a clear case of high bias.")
print("- LogisticRegression here uses ALL FOUR features and LEARNS its own coefficients,")
print("  reaching accuracy close to 1.0 on data it has never seen during training.")
print("Letting the model adjust itself to richer features is the single biggest reason")
print("its error is so much lower than our manual formula's error.")

## 7. Data Visualization

### WHY?
Visualizations help us identify patterns and relationships within the features, such as feature correlation or cluster separation, showing how easily the target classes can be distinguished.

### HOW?
We use Seaborn and Matplotlib to plot target label distribution and visualize a scatter plot comparing sepal length and petal length.

In [ ]:
# Set up figure canvas for side-by-side plots
plt.figure(figsize=(12, 5))

# Plot target class distribution to audit balance
plt.subplot(1, 2, 1)
sns.countplot(x=y, hue=y, legend=False, palette='viridis')
plt.xticks(ticks=[0, 1, 2], labels=iris.target_names)
plt.title('Target Class Distribution')
plt.xlabel('Species')
plt.ylabel('Count')

# Visualize cluster separation comparing sepal vs. petal length
plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='sepal length (cm)', y='petal length (cm)', hue='target', palette='viridis')
plt.title('Sepal vs Petal Length Cluster Separation')
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Petal Length (cm)')
plt.legend(title='Species', labels=iris.target_names.tolist())

plt.tight_layout()
plt.show()

## 8. Part 4: Evaluating Validation Splits

### WHY?
We analyze how varying the test split size ($20\%$, $30\%$, $40\%$) affects classification accuracy on unseen test data. This helps us see if changing data volume shifts our model toward underfitting.

### HOW?
We loop through test split ratios, fit `LogisticRegression` for each configuration, and compile validation accuracies in a summary table.

In [ ]:
# Collect (test_size, accuracy) pairs as we loop
results = []

# Try three different test split sizes
for ts in [0.2, 0.3, 0.4]:
    # Split data; random_state stays fixed to ensure a fair comparison
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=ts, random_state=42)

    # Fit a new Logistic Regression model
    model = LogisticRegression(max_iter=200)
    model.fit(X_tr, y_tr)

    # Generate predictions and compute accuracy
    pred = model.predict(X_te)
    a = accuracy_score(y_te, pred)

    results.append((ts, a))

# Display results in a dataframe table
results_df = pd.DataFrame(results, columns=['test_size', 'accuracy'])
print("Comparison of test_size vs accuracy:")
print(results_df)

# Print discussion
print("\nWhat this suggests about bias:")
print("Across test_size values of 0.2, 0.3, and 0.4, accuracy on Iris stays high and")
print("barely moves. LogisticRegression with all four features already has enough")
print("flexibility to capture the pattern in this dataset, so giving it a slightly")
print("smaller training set doesn't push it toward underfitting here.")
print("For datasets that are harder to separate, you would typically see accuracy drop")
print("more noticeably as test_size grows - and techniques like k-fold cross-validation")
print("(e.g. scikit-learn's cross_val_score) give an even more reliable picture by")
print("averaging results across several different splits instead of just one.")

# Part 5: Interview Corner

**Q1. In your own words, what is bias?**
Bias is the systematic error introduced when a model's assumptions are too simple to represent the true patterns in the data. It causes the model's predictions to be consistently off-target in a predictable direction, leading to underfitting.

**Q2. Why does bias happen in the first place?**
Usually, it is due to one or more of the following: the model is too simple for the problem (e.g., using a straight line to fit a curved relationship), key features with strong predictive signal were left out, or the parameters were not allowed to adjust to the data (as in our manual NumPy model).

**Q3. Why does high bias lead to underfitting?**
Because the model's assumptions limit how well it can represent the data. It hits a performance ceiling on the *training* data itself, not just on unseen validation data. Poor performance across both splits is the classic signature of underfitting.

**Q4. How would you explain bias vs. variance to someone new to ML?**
Picture shooting free throws. High bias is like always aiming slightly to the left of the hoop—you are consistent, but consistently wrong. High variance is like your shots being scattered wildly all over the backboard—unpredictable and highly variable. The goal is to aim straight and hit the target consistently.

**Q5. What are practical ways to reduce bias?**
Use a more complex model (e.g., move from a single linear feature to a multi-feature classifier), add informative features that carry real signal, reduce regularization strength, and make sure training is allowed to run long enough (sufficient `max_iter`) to converge.

**Q6. Why bother with a train/test split at all?**
It is the simplest way to check whether a model has learned generalizable patterns or is simply memorizing training noise. Without it, an underfit (high-bias) model and an overfit (high-variance) model could both appear performant on the training data alone.

**Q7. How does feature engineering connect to bias?**
Features are the primary signal the model has to work with. In this notebook, moving from one feature (sepal length only) to all four measurements was the single biggest factor in closing the gap between our high-bias manual model and the trained `LogisticRegression` model.

# Key Takeaways

- **Bias is a consistent, systematic error** caused by a model's oversimplified assumptions—creating a predictable blind spot rather than random noise.
- **Underfitting is bias made visible**: a high-bias model performs poorly even on the training data it was fitted on, because its structure was never flexible enough to represent the underlying patterns.
- **The bias-variance tradeoff is a balancing act**: increasing model flexibility reduces bias but risks raising variance (overfitting). The goal is to find the sweet spot that minimizes total error.
- **Feature count and parameter flexibility are key**: our hand-tuned single-feature model produced a large, unmovable MSE, whereas a `LogisticRegression` model using all four features and optimizing its own parameters reached near-perfect accuracy.